# Gesture Synth Experiments

Use this notebook to test oscillator waveforms, play the configured synth notes, and debug webcam hand tracking without running the desktop app.

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.config import AppConfig, SynthConfig
from src.synth import Synthesizer

config = AppConfig.load(PROJECT_ROOT / 'config.json')
config

## Oscillator generation

In [ ]:
for waveform in ['sine', 'square', 'sawtooth', 'triangle']:
    synth = Synthesizer(SynthConfig(sample_rate=44100, waveform=waveform, attack=0.001))
    synth.note_on('C4', 261.63)
    audio = synth.render(1200)
    plt.figure(figsize=(10, 2.5))
    plt.plot(audio[:400])
    plt.title(waveform)
    plt.ylim(-0.35, 0.35)
    plt.grid(True, alpha=0.25)
    plt.show()

## Play the synth through your speakers

This uses the same real-time `sounddevice` synth engine as `main.py`, not audio files. Run `audition_gesture_sound(1)` for one note or run the final line to hear all five configured sounds.

In [ ]:
def audition_gesture_sound(fingers: int, seconds: float = 0.65) -> None:
    """Play the note assigned to a finger-count gesture."""
    note = config.gesture_notes[fingers]
    live_synth = Synthesizer(config.synth)
    live_synth.start()
    try:
        print(f'Playing {fingers} fingers: {note.name} ({note.frequency:.2f} Hz)')
        live_synth.note_on(note.name, note.frequency)
        time.sleep(seconds)
        live_synth.note_off()
        time.sleep(config.synth.release + 0.05)
    finally:
        live_synth.close()

# Run one of these independently:
# audition_gesture_sound(1)
# audition_gesture_sound(2)

for fingers in sorted(config.gesture_notes):
    audition_gesture_sound(fingers, seconds=0.45)

## Webcam hand landmarks with live sound

Run this cell locally with a webcam. The OpenCV preview draws landmarks and plays a stable 1-5 finger gesture through the same synth engine. Press `q` in the preview window to stop.

In [ ]:
import cv2

from src.camera import Camera
from src.gesture_detector import GestureStabilizer, count_extended_fingers, supported_gesture
from src.hand_tracker import HandTracker
from src.ui import FPSCounter, draw_overlay

camera = Camera(config.camera)
tracker = HandTracker(config.gesture)
synth = Synthesizer(config.synth)
stabilizer = GestureStabilizer(config.gesture.stable_frames)
fps = FPSCounter()
last_note_name = None
synth.start()

try:
    while True:
        frame = camera.read()
        if frame is None:
            if cv2.waitKey(30) & 0xFF in (ord('q'), 27):
                break
            continue
        if config.gesture.mirror_camera:
            frame = cv2.flip(frame, 1)
        detection, results = tracker.process(frame)
        raw = None
        if detection:
            raw = count_extended_fingers(detection.landmarks.landmark, detection.handedness, mirrored=config.gesture.mirror_camera)
        state = stabilizer.update(supported_gesture(raw, config.gesture_notes.keys()))
        note = config.gesture_notes.get(state.stable_fingers)
        if note is None and last_note_name is not None:
            synth.note_off()
            last_note_name = None
        elif note is not None and note.name != last_note_name:
            synth.note_on(note.name, note.frequency)
            last_note_name = note.name
        tracker.draw(frame, results)
        draw_overlay(frame, state, note, config.synth.waveform, fps.update())
        cv2.imshow('Gesture Debug', frame)
        if cv2.waitKey(1) & 0xFF in (ord('q'), 27):
            break
finally:
    synth.note_off()
    synth.close()
    tracker.close()
    camera.close()
    cv2.destroyAllWindows()